In [88]:
import pandas as pd
from pathlib import Path

In [89]:
PROCESSED_DATA_DIR = Path("../data/processed")
SCORES_DATA_DIR = Path("../data/scores")
OUTPUTS_DATA_DIR = Path("../data/outputs") 

In [90]:
observed_time_weights = {
    "recent": 0.60,
    "historical": 0.40,
}

minimum_coverage_target = 0.90
recommended_coverage_target = 0.99

excluded_statuses = ["Cancelada"]

ANALYSIS_DATE = pd.Timestamp("2026-07-29").normalize()

recent_start = (
    ANALYSIS_DATE
    - pd.DateOffset(months=12)
    + pd.Timedelta(days=1)
)

assert abs(
    sum(observed_time_weights.values()) - 1
) < 1e-9, (
    "Observed-period weights must sum to 1."
)

assert minimum_coverage_target < recommended_coverage_target, (
    "The minimum coverage target must be lower "
    "than the recommended target."
)

ANALYSIS_DATE, recent_start

(Timestamp('2026-07-29 00:00:00'), Timestamp('2025-07-30 00:00:00'))

In [91]:
info_actions_df = pd.read_csv(
    PROCESSED_DATA_DIR / "info_actions.csv",
    sep=",",
    encoding="utf-8"
)

actions_df = pd.read_csv(
    PROCESSED_DATA_DIR / "training_actions.csv",
    sep=",",
    encoding="utf-8",
    parse_dates=[
        "start_date",
        "end_date"
    ]
)

trainers_course_df = pd.read_csv(
    PROCESSED_DATA_DIR / "trainers_course.csv",
    sep=",",
    encoding="utf-8"
)

trainers_local_df = pd.read_csv(
    PROCESSED_DATA_DIR / "trainers_local.csv",
    sep=",",
    encoding="utf-8"
)

priority_df = pd.read_csv(
    OUTPUTS_DATA_DIR / "priority.csv",
    sep=",",
    encoding="utf-8"
)

## Course-local combinations and current trainer capacity

In [92]:
active_courses_df = (
    info_actions_df
    .loc[
        info_actions_df["active"],
        [
            "course_id",
            "course_name",
            "course_area"
        ]
    ]
    .drop_duplicates()
)

locals_df = (
    actions_df[
        ["local"]
    ]
    .drop_duplicates()
    .sort_values("local")
)

course_local_df = (
    active_courses_df
    .merge(
        locals_df,
        how="cross"
    )
)

In [93]:
trainer_eligibility_df = (
    trainers_course_df
    .merge(
        trainers_local_df,
        how="inner",
        on="trainer_id",
        validate="many_to_many"
    )
)

current_trainer_count_df = (
    trainer_eligibility_df
    .groupby(
        [
            "course_id",
            "local"
        ],
        as_index=False
    )
    .agg(
        current_trainer_count=(
            "trainer_id",
            "nunique"
        )
    )
)

course_local_df = (
    course_local_df
    .merge(
        current_trainer_count_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .fillna({
        "current_trainer_count": 0
    })
)

course_local_df["current_trainer_count"] = (
    course_local_df["current_trainer_count"]
    .astype(int)
)

course_local_df

,course_id,course_name,course_area,local,current_trainer_count
0,CYB,Cibersegurança Básica,Informática,Centro 1,3
1,CYB,Cibersegurança Básica,Informática,Centro 2,3
2,CYB,Cibersegurança Básica,Informática,Centro 3,4
3,CYB,Cibersegurança Básica,Informática,Centro 4,4
4,CYB,Cibersegurança Básica,Informática,Centro 5,4
...,...,...,...,...,...
115,VND,Vendas e Negociação,Comercial,Centro 1,4
116,VND,Vendas e Negociação,Comercial,Centro 2,4
117,VND,Vendas e Negociação,Comercial,Centro 3,4
118,VND,Vendas e Negociação,Comercial,Centro 4,2


## Daily operational demand

Cancelled actions are excluded. The remaining actions are expanded into one row per active calendar day.

In [94]:
capacity_actions_df = (
    actions_df
    .loc[
        ~actions_df["status"]
        .isin(excluded_statuses)
    ]
    .copy()
)

future_end = (
    capacity_actions_df["end_date"]
    .max()
    .normalize()
)

print(
    "Historical period:",
    capacity_actions_df["start_date"].min().date(),
    "-",
    (recent_start - pd.Timedelta(days=1)).date()
)

print(
    "Recent period:",
    recent_start.date(),
    "-",
    ANALYSIS_DATE.date()
)

print(
    "Future period:",
    (ANALYSIS_DATE + pd.Timedelta(days=1)).date(),
    "-",
    future_end.date()
)

Historical period: 2018-01-01 - 2025-07-29
Recent period: 2025-07-30 - 2026-07-29
Future period: 2026-07-30 - 2026-12-31


In [95]:
capacity_actions_df["date"] = (
    capacity_actions_df
    .apply(
        lambda row:
            pd.date_range(
                start=row["start_date"],
                end=row["end_date"],
                freq="D"
            ),
        axis=1
    )
)

action_days_df = (
    capacity_actions_df[
        [
            "action_id",
            "course_id",
            "local",
            "date"
        ]
    ]
    .explode(
        "date",
        ignore_index=True
    )
)

daily_demand_df = (
    action_days_df
    .groupby(
        [
            "course_id",
            "local",
            "date"
        ],
        as_index=False
    )
    .agg(
        active_actions=(
            "action_id",
            "nunique"
        )
    )
)

daily_demand_df

,course_id,local,date,active_actions
0,CYB,Centro 1,2018-01-10,1
1,CYB,Centro 1,2018-01-11,1
2,CYB,Centro 1,2018-01-12,1
3,CYB,Centro 1,2018-01-13,1
4,CYB,Centro 1,2018-01-14,1
...,...,...,...,...
83269,VND,Centro 5,2026-12-04,2
83270,VND,Centro 5,2026-12-05,2
83271,VND,Centro 5,2026-12-06,2
83272,VND,Centro 5,2026-12-07,2


## Effective shared trainer capacity

When the same trainer can cover several active demands on the same date, their single daily capacity is divided proportionally across those demands.

In [96]:
trainer_daily_demand_df = (
    daily_demand_df
    .merge(
        trainer_eligibility_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="many_to_many"
    )
)

assert trainer_daily_demand_df["trainer_id"].notna().all(), (
    "At least one active course-local combination has no eligible trainer."
)

trainer_daily_demand_df["trainer_total_active_actions"] = (
    trainer_daily_demand_df
    .groupby(
        [
            "trainer_id",
            "date"
        ]
    )["active_actions"]
    .transform("sum")
)

trainer_daily_demand_df["trainer_capacity_share"] = (
    trainer_daily_demand_df["active_actions"]
    .div(
        trainer_daily_demand_df[
            "trainer_total_active_actions"
        ]
    )
)

In [97]:
daily_capacity_df = (
    trainer_daily_demand_df
    .groupby(
        [
            "course_id",
            "local",
            "date"
        ],
        as_index=False
    )
    .agg(
        active_actions=(
            "active_actions",
            "first"
        ),

        effective_trainer_capacity=(
            "trainer_capacity_share",
            "sum"
        ),

        eligible_trainers=(
            "trainer_id",
            "nunique"
        )
    )
    .assign(
        capacity_gap=lambda df:
            df["active_actions"]
            .sub(
                df[
                    "effective_trainer_capacity"
                ]
            )
            .clip(lower=0)
    )
)

capacity_gap_floor = (
    daily_capacity_df["capacity_gap"]
    .astype(int)
)

daily_capacity_df[
    "additional_trainers_needed_day"
] = (
    capacity_gap_floor
    + daily_capacity_df["capacity_gap"]
    .gt(capacity_gap_floor)
    .astype(int)
)

daily_capacity_df["period"] = "historical"

recent_mask = (
    daily_capacity_df["date"]
    .between(
        recent_start,
        ANALYSIS_DATE,
        inclusive="both"
    )
)

future_mask = (
    daily_capacity_df["date"]
    .gt(ANALYSIS_DATE)
)

daily_capacity_df.loc[
    recent_mask,
    "period"
] = "recent"

daily_capacity_df.loc[
    future_mask,
    "period"
] = "future"

daily_capacity_df["period"] = pd.Categorical(
    daily_capacity_df["period"],
    categories=[
        "historical",
        "recent",
        "future",
    ],
    ordered=True
)

daily_capacity_df["period"].value_counts().sort_index()

period
historical    63444
recent        13924
future         5906
Name: count, dtype: int64

## Demand and shortage summary

In [98]:
period_summary_df = (
    daily_capacity_df
    .groupby(
        [
            "course_id",
            "local",
            "period",
        ],
        as_index=False,
        observed=True
    )
    .agg(
        active_days=(
            "date",
            "nunique"
        ),

        average_active_actions=(
            "active_actions",
            "mean"
        ),

        peak_active_actions=(
            "active_actions",
            "max"
        ),

        average_effective_capacity=(
            "effective_trainer_capacity",
            "mean"
        ),

        peak_additional_trainers_needed=(
            "additional_trainers_needed_day",
            "max"
        )
    )
)

period_summary_wide_df = (
    period_summary_df
    .pivot(
        index=[
            "course_id",
            "local",
        ],
        columns="period"
    )
)

period_summary_wide_df.columns = [
    f"{metric}_{period}"
    for metric, period
    in period_summary_wide_df.columns
]

period_summary_wide_df = (
    period_summary_wide_df
    .reset_index()
    .fillna(0)
)

period_summary_wide_df

,course_id,local,active_days_historical,active_days_recent,active_days_future,average_active_actions_historical,average_active_actions_recent,average_active_actions_future,peak_active_actions_historical,peak_active_actions_recent,peak_active_actions_future,average_effective_capacity_historical,average_effective_capacity_recent,average_effective_capacity_future,peak_additional_trainers_needed_historical,peak_additional_trainers_needed_recent,peak_additional_trainers_needed_future
0,CYB,Centro 1,680,231,57,1.191176,1.502165,1.684211,4,4,4,1.769526,1.444629,1.353294,3,3,2
1,CYB,Centro 2,873,199,79,1.324170,1.663317,1.974684,6,5,5,2.471581,2.061047,2.367294,4,3,4
2,CYB,Centro 3,664,178,73,1.165663,1.410112,1.452055,3,5,5,2.578173,2.241342,2.487541,2,3,3
3,CYB,Centro 4,956,198,115,1.311715,1.727273,1.556522,4,4,5,2.785265,2.460961,2.194763,2,3,3
4,CYB,Centro 5,567,124,47,1.231041,1.532258,1.702128,5,5,4,3.030309,2.394720,2.530674,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,VND,Centro 1,466,67,49,1.126609,1.119403,1.081633,3,2,2,2.967675,2.507191,2.651790,1,0,0
116,VND,Centro 2,654,122,80,1.165138,1.327869,1.725000,4,3,4,3.069703,2.622717,2.831637,1,1,2
117,VND,Centro 3,371,87,37,1.088949,1.183908,1.108108,3,3,2,3.043822,2.569966,2.264996,0,1,0
118,VND,Centro 4,551,142,52,1.230490,1.211268,1.365385,4,3,3,1.244406,1.081221,1.109936,3,2,2


## Coverage simulation

The simulation tests every feasible number of dedicated additional trainers. The first value reaching each coverage target becomes the corresponding recommendation.

In [99]:
max_additional_trainers = int(
    daily_capacity_df[
        "additional_trainers_needed_day"
    ]
    .max()
)

additional_candidates_df = pd.DataFrame({
    "additional_trainers":
        range(max_additional_trainers + 1)
})

coverage_simulation_df = (
    daily_capacity_df[
        [
            "course_id",
            "local",
            "period",
            "additional_trainers_needed_day",
        ]
    ]
    .merge(
        additional_candidates_df,
        how="cross"
    )
    .assign(
        covered=lambda df:
            df["additional_trainers"]
            .ge(
                df[
                    "additional_trainers_needed_day"
                ]
            )
    )
    .groupby(
        [
            "course_id",
            "local",
            "additional_trainers",
            "period",
        ],
        as_index=False,
        observed=True
    )
    .agg(
        coverage_rate=(
            "covered",
            "mean"
        )
    )
)

coverage_simulation_df["coverage_rate"] = (
    coverage_simulation_df["coverage_rate"]
    .mul(100)
)

In [100]:
coverage_simulation_wide_df = (
    coverage_simulation_df
    .pivot(
        index=[
            "course_id",
            "local",
            "additional_trainers",
        ],
        columns="period",
        values="coverage_rate"
    )
    .reset_index()
    .rename(
        columns={
            "historical":
                "coverage_rate_historical",

            "recent":
                "coverage_rate_recent",

            "future":
                "coverage_rate_future",
        }
    )
)

coverage_simulation_wide_df.columns.name = None

period_coverage_columns = [
    "coverage_rate_historical",
    "coverage_rate_recent",
    "coverage_rate_future",
]

for column in period_coverage_columns:
    if column not in coverage_simulation_wide_df:
        coverage_simulation_wide_df[column] = 100.0

coverage_simulation_wide_df[
    period_coverage_columns
] = (
    coverage_simulation_wide_df[
        period_coverage_columns
    ]
    .fillna(100.0)
)

In [101]:
coverage_simulation_wide_df[
    "observed_weighted_coverage_rate"
] = (
    coverage_simulation_wide_df[
        "coverage_rate_recent"
    ]
    .mul(
        observed_time_weights["recent"]
    )
    .add(
        coverage_simulation_wide_df[
            "coverage_rate_historical"
        ]
        .mul(
            observed_time_weights["historical"]
        )
    )
)

coverage_simulation_wide_df[
    "planning_coverage_rate"
] = (
    coverage_simulation_wide_df[
        [
            "observed_weighted_coverage_rate",
            "coverage_rate_future",
        ]
    ]
    .min(axis=1)
)

coverage_simulation_wide_df

,course_id,local,additional_trainers,coverage_rate_historical,coverage_rate_recent,coverage_rate_future,observed_weighted_coverage_rate,planning_coverage_rate
0,CYB,Centro 1,0,74.558824,44.588745,26.315789,56.576776,26.315789
1,CYB,Centro 1,1,98.088235,88.311688,82.456140,92.222307,82.456140
2,CYB,Centro 1,2,99.705882,98.701299,100.000000,99.103132,99.103132
3,CYB,Centro 1,3,100.000000,100.000000,100.000000,100.000000,100.000000
4,CYB,Centro 1,4,100.000000,100.000000,100.000000,100.000000,100.000000
...,...,...,...,...,...,...,...,...
835,VND,Centro 5,2,100.000000,100.000000,100.000000,100.000000,100.000000
836,VND,Centro 5,3,100.000000,100.000000,100.000000,100.000000,100.000000
837,VND,Centro 5,4,100.000000,100.000000,100.000000,100.000000,100.000000
838,VND,Centro 5,5,100.000000,100.000000,100.000000,100.000000,100.000000


In [102]:
def select_trainer_requirement(
    coverage_df,
    coverage_target,
    output_name,
):
    selected_df = (
        coverage_df
        .loc[
            coverage_df[
                "planning_coverage_rate"
            ]
            .ge(
                coverage_target * 100
            )
        ]
        .sort_values(
            [
                "course_id",
                "local",
                "additional_trainers",
            ]
        )
        .drop_duplicates(
            [
                "course_id",
                "local",
            ]
        )
        [
            [
                "course_id",
                "local",
                "additional_trainers",

                "coverage_rate_recent",
                "coverage_rate_historical",
                "coverage_rate_future",

                "observed_weighted_coverage_rate",
                "planning_coverage_rate",
            ]
        ]
    )

    return selected_df.rename(
        columns={
            "additional_trainers":
                f"{output_name}_additional_trainers",

            "coverage_rate_recent":
                f"{output_name}_coverage_rate_recent",

            "coverage_rate_historical":
                f"{output_name}_coverage_rate_historical",

            "coverage_rate_future":
                f"{output_name}_coverage_rate_future",

            "observed_weighted_coverage_rate":
                f"{output_name}_observed_weighted_coverage_rate",

            "planning_coverage_rate":
                f"{output_name}_planning_coverage_rate",
        }
    )

In [103]:
current_coverage_df = (
    coverage_simulation_wide_df
    .loc[
        coverage_simulation_wide_df[
            "additional_trainers"
        ]
        .eq(0),
        [
            "course_id",
            "local",

            "coverage_rate_recent",
            "coverage_rate_historical",
            "coverage_rate_future",

            "observed_weighted_coverage_rate",
            "planning_coverage_rate",
        ],
    ]
    .rename(
        columns={
            "coverage_rate_recent":
                "current_coverage_rate_recent",

            "coverage_rate_historical":
                "current_coverage_rate_historical",

            "coverage_rate_future":
                "current_coverage_rate_future",

            "observed_weighted_coverage_rate":
                "current_observed_weighted_coverage_rate",

            "planning_coverage_rate":
                "current_planning_coverage_rate",
        }
    )
)

minimum_requirement_df = (
    select_trainer_requirement(
        coverage_simulation_wide_df,
        minimum_coverage_target,
        "minimum"
    )
)

recommended_requirement_df = (
    select_trainer_requirement(
        coverage_simulation_wide_df,
        recommended_coverage_target,
        "recommended"
    )
)

## Final trainer requirements

In [104]:
priority_columns = [
    "course_id",
    "local",
    "priority_score",
    "priority_level",
    "priority_assessment",
]

trainer_requirements_df = (
    course_local_df
    .merge(
        period_summary_wide_df,
        how="left",
        on=[
            "course_id",
            "local",
        ],
        validate="one_to_one"
    )
    .merge(
        current_coverage_df,
        how="left",
        on=[
            "course_id",
            "local",
        ],
        validate="one_to_one"
    )
    .merge(
        minimum_requirement_df,
        how="left",
        on=[
            "course_id",
            "local",
        ],
        validate="one_to_one"
    )
    .merge(
        recommended_requirement_df,
        how="left",
        on=[
            "course_id",
            "local",
        ],
        validate="one_to_one"
    )
    .merge(
        priority_df[
            priority_columns
        ],
        how="left",
        on=[
            "course_id",
            "local",
        ],
        validate="one_to_one"
    )
)

In [105]:
additional_trainer_columns = [
    "minimum_additional_trainers",
    "recommended_additional_trainers",
]

trainer_requirements_df[
    additional_trainer_columns
] = (
    trainer_requirements_df[
        additional_trainer_columns
    ]
    .fillna(0)
    .astype(int)
)

coverage_columns = [
    column
    for column
    in trainer_requirements_df.columns
    if "coverage_rate" in column
]

trainer_requirements_df[
    coverage_columns
] = (
    trainer_requirements_df[
        coverage_columns
    ]
    .fillna(100)
    .round(2)
)

demand_count_columns = [
    column
    for column
    in trainer_requirements_df.columns
    if column.startswith((
        "active_days_",
        "peak_active_actions_",
        "peak_additional_trainers_needed_",
    ))
]

trainer_requirements_df[
    demand_count_columns
] = (
    trainer_requirements_df[
        demand_count_columns
    ]
    .fillna(0)
    .astype(int)
)

In [106]:
trainer_requirements_df = (
    trainer_requirements_df
    .assign(
        minimum_total_trainers=lambda df:
            df["current_trainer_count"]
            .add(
                df[
                    "minimum_additional_trainers"
                ]
            ),

        recommended_total_trainers=lambda df:
            df["current_trainer_count"]
            .add(
                df[
                    "recommended_additional_trainers"
                ]
            )
    )
)

In [107]:
trainer_requirements_df = (
    trainer_requirements_df[
        [
            "course_id",
            "course_name",
            "course_area",
            "local",

            "priority_score",
            "priority_level",
            "priority_assessment",

            "current_trainer_count",

            "minimum_additional_trainers",
            "recommended_additional_trainers",

            "minimum_total_trainers",
            "recommended_total_trainers",

            "current_planning_coverage_rate",
            "minimum_planning_coverage_rate",
            "recommended_planning_coverage_rate",

            "current_observed_weighted_coverage_rate",
            "minimum_observed_weighted_coverage_rate",
            "recommended_observed_weighted_coverage_rate",

            "current_coverage_rate_future",
            "minimum_coverage_rate_future",
            "recommended_coverage_rate_future",

            "current_coverage_rate_recent",
            "current_coverage_rate_historical",

            "active_days_future",
            "active_days_recent",
            "active_days_historical",

            "peak_active_actions_future",
            "peak_active_actions_recent",
            "peak_active_actions_historical",

            "peak_additional_trainers_needed_future",
            "peak_additional_trainers_needed_recent",
            "peak_additional_trainers_needed_historical",
        ]
    ]
    .sort_values(
        [
            "recommended_additional_trainers",
            "priority_score",
            "current_planning_coverage_rate",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        ignore_index=True
    )
)

trainer_requirements_df

,course_id,course_name,course_area,local,priority_score,priority_level,priority_assessment,current_trainer_count,minimum_additional_trainers,recommended_additional_trainers,...,current_coverage_rate_historical,active_days_future,active_days_recent,active_days_historical,peak_active_actions_future,peak_active_actions_recent,peak_active_actions_historical,peak_additional_trainers_needed_future,peak_additional_trainers_needed_recent,peak_additional_trainers_needed_historical
0,PBI,Power BI e Visualização de Dados,Informática,Centro 2,80.72,High,High priority with moderate agreement,4,2,4,...,95.24,97,291,1072,6,7,5,4,4,2
1,CYB,Cibersegurança Básica,Informática,Centro 2,77.65,High,High priority with moderate agreement,3,2,4,...,94.50,79,199,873,5,5,6,4,3,4
2,PYT,Introdução à Programação em Python,Informática,Centro 4,56.30,Upper-middle,Moderate priority,3,2,4,...,94.15,77,144,854,5,4,4,4,3,2
3,PYT,Introdução à Programação em Python,Informática,Centro 5,45.30,Lower-middle,Low priority,4,3,4,...,95.07,53,194,467,6,3,3,4,1,2
4,LOG,Logística e Gestão de Armazém,Logística,Centro 5,81.44,High,Confirmed high priority,2,2,3,...,84.24,44,119,495,3,3,3,3,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 3,22.82,Low,Low priority,4,0,0,...,100.00,29,62,283,2,3,2,0,0,0
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,22.72,Low,Low priority,2,0,0,...,100.00,21,54,192,1,1,2,0,0,0
117,VND,Vendas e Negociação,Comercial,Centro 3,21.93,Low,Low priority,4,0,0,...,100.00,37,87,371,2,3,3,0,1,0
118,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,21.62,Low,Low priority,4,0,0,...,100.00,16,54,307,1,3,2,0,0,0


In [108]:
expected_combinations = len(
    course_local_df
)

assert len(trainer_requirements_df) == expected_combinations, (
    "Some course-local combinations were lost."
)

assert not trainer_requirements_df.duplicated(
    [
        "course_id",
        "local",
    ]
).any(), (
    "Duplicate course-local combinations found."
)

assert trainer_requirements_df[
    "recommended_additional_trainers"
].ge(
    trainer_requirements_df[
        "minimum_additional_trainers"
    ]
).all(), (
    "Recommended additional trainers cannot "
    "be lower than the minimum."
)

assert trainer_requirements_df[
    "recommended_total_trainers"
].ge(
    trainer_requirements_df[
        "current_trainer_count"
    ]
).all(), (
    "Recommended total trainers cannot be "
    "lower than the current count."
)

assert trainer_requirements_df[
    "recommended_planning_coverage_rate"
].ge(
    recommended_coverage_target * 100
).all(), (
    "At least one recommendation does not "
    "reach the planning coverage target."
)

print("Capacity validation passed.")

Capacity validation passed.


In [109]:
priority_capacity_summary = (
    trainer_requirements_df
    .groupby(
        "priority_level",
        observed=True
    )
    .agg(
        combinations=(
            "course_id",
            "size"
        ),

        average_priority_score=(
            "priority_score",
            "mean"
        ),

        average_current_planning_coverage=(
            "current_planning_coverage_rate",
            "mean"
        ),

        average_additional_trainers=(
            "recommended_additional_trainers",
            "mean"
        ),

        median_additional_trainers=(
            "recommended_additional_trainers",
            "median"
        ),

        maximum_additional_trainers=(
            "recommended_additional_trainers",
            "max"
        )
    )
    .round(2)
)

priority_capacity_summary

,combinations,average_priority_score,average_current_planning_coverage,average_additional_trainers,median_additional_trainers,maximum_additional_trainers
priority_level,,,,,,
High,11,79.87,64.04,2.91,3.0,4
Low,10,21.65,88.50,0.30,0.0,2
Lower-middle,52,39.13,92.37,0.92,1.0,4
Upper-middle,47,61.93,79.29,2.02,2.0,4


In [110]:
priority_capacity_correlation = (
    trainer_requirements_df[
        "priority_score"
    ]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "recommended_additional_trainers"
        ]
        .rank(method="average")
    )
)

priority_coverage_correlation = (
    trainer_requirements_df[
        "priority_score"
    ]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "current_planning_coverage_rate"
        ]
        .rank(method="average")
    )
)

print(
    "Priority-capacity correlation:",
    priority_capacity_correlation.round(3)
)

print(
    "Priority-coverage correlation:",
    priority_coverage_correlation.round(3)
)

Priority-capacity correlation: 0.763
Priority-coverage correlation: -0.715


## Validation

The recommendation must reach the selected coverage target for every combination represented in the operational history.

In [111]:
expected_combinations = len(
    course_local_df
)

assert len(trainer_requirements_df) == expected_combinations, (
    "Some course-local combinations were lost."
)

assert not trainer_requirements_df.duplicated(
    [
        "course_id",
        "local",
    ]
).any(), (
    "Duplicate course-local combinations found."
)

assert trainer_requirements_df[
    "recommended_additional_trainers"
].ge(
    trainer_requirements_df[
        "minimum_additional_trainers"
    ]
).all(), (
    "Recommended additional trainers cannot "
    "be lower than the minimum."
)

assert trainer_requirements_df[
    "recommended_total_trainers"
].ge(
    trainer_requirements_df[
        "current_trainer_count"
    ]
).all(), (
    "Recommended total trainers cannot be "
    "lower than the current count."
)

assert trainer_requirements_df[
    "recommended_planning_coverage_rate"
].ge(
    recommended_coverage_target * 100
).all(), (
    "At least one recommendation does not "
    "reach the planning coverage target."
)

print("Capacity validation passed.")

Capacity validation passed.


In [112]:
trainer_requirements_df[
    "minimum_additional_trainers"
].value_counts().sort_index().reset_index()

,minimum_additional_trainers,count
0,0,57
1,1,47
2,2,15
3,3,1


In [113]:
trainer_requirements_df[
    "recommended_additional_trainers"
].value_counts().sort_index().reset_index()

,recommended_additional_trainers,count
0,0,24
1,1,41
2,2,32
3,3,19
4,4,4


In [114]:
priority_capacity_summary = (
    trainer_requirements_df
    .groupby(
        "priority_level",
        observed=True
    )
    .agg(
        combinations=(
            "course_id",
            "size"
        ),

        average_priority_score=(
            "priority_score",
            "mean"
        ),

        average_current_planning_coverage=(
            "current_planning_coverage_rate",
            "mean"
        ),

        average_additional_trainers=(
            "recommended_additional_trainers",
            "mean"
        ),

        median_additional_trainers=(
            "recommended_additional_trainers",
            "median"
        ),

        maximum_additional_trainers=(
            "recommended_additional_trainers",
            "max"
        )
    )
    .round(2)
)

priority_capacity_summary

,combinations,average_priority_score,average_current_planning_coverage,average_additional_trainers,median_additional_trainers,maximum_additional_trainers
priority_level,,,,,,
High,11,79.87,64.04,2.91,3.0,4
Low,10,21.65,88.50,0.30,0.0,2
Lower-middle,52,39.13,92.37,0.92,1.0,4
Upper-middle,47,61.93,79.29,2.02,2.0,4


In [115]:
priority_capacity_correlation = (
    trainer_requirements_df["priority_score"]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "recommended_additional_trainers"
        ]
        .rank(method="average")
    )
)

print(priority_capacity_correlation.round(3))

0.763


In [116]:
priority_coverage_correlation = (
    trainer_requirements_df[
        "priority_score"
    ]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "current_planning_coverage_rate"
        ]
        .rank(method="average")
    )
)

print(priority_coverage_correlation.round(3))

-0.715


In [117]:
trainer_requirements_df = (
    trainer_requirements_df[
        [
            "course_id",
            "course_name",
            "course_area",
            "local",

            "priority_score",
            "priority_level",
            "priority_assessment",

            "current_trainer_count",

            "minimum_additional_trainers",
            "recommended_additional_trainers",

            "minimum_total_trainers",
            "recommended_total_trainers",

            "current_planning_coverage_rate",
            "minimum_planning_coverage_rate",
            "recommended_planning_coverage_rate",

            "current_observed_weighted_coverage_rate",
            "minimum_observed_weighted_coverage_rate",
            "recommended_observed_weighted_coverage_rate",

            "current_coverage_rate_future",
            "minimum_coverage_rate_future",
            "recommended_coverage_rate_future",

            "current_coverage_rate_recent",
            "current_coverage_rate_historical",

            "active_days_future",
            "active_days_recent",
            "active_days_historical",

            "peak_active_actions_future",
            "peak_active_actions_recent",
            "peak_active_actions_historical",

            "peak_additional_trainers_needed_future",
            "peak_additional_trainers_needed_recent",
            "peak_additional_trainers_needed_historical",
        ]
    ]
    .sort_values(
        [
            "recommended_additional_trainers",
            "priority_score",
            "current_planning_coverage_rate",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        ignore_index=True
    )
)

trainer_requirements_df

,course_id,course_name,course_area,local,priority_score,priority_level,priority_assessment,current_trainer_count,minimum_additional_trainers,recommended_additional_trainers,...,current_coverage_rate_historical,active_days_future,active_days_recent,active_days_historical,peak_active_actions_future,peak_active_actions_recent,peak_active_actions_historical,peak_additional_trainers_needed_future,peak_additional_trainers_needed_recent,peak_additional_trainers_needed_historical
0,PBI,Power BI e Visualização de Dados,Informática,Centro 2,80.72,High,High priority with moderate agreement,4,2,4,...,95.24,97,291,1072,6,7,5,4,4,2
1,CYB,Cibersegurança Básica,Informática,Centro 2,77.65,High,High priority with moderate agreement,3,2,4,...,94.50,79,199,873,5,5,6,4,3,4
2,PYT,Introdução à Programação em Python,Informática,Centro 4,56.30,Upper-middle,Moderate priority,3,2,4,...,94.15,77,144,854,5,4,4,4,3,2
3,PYT,Introdução à Programação em Python,Informática,Centro 5,45.30,Lower-middle,Low priority,4,3,4,...,95.07,53,194,467,6,3,3,4,1,2
4,LOG,Logística e Gestão de Armazém,Logística,Centro 5,81.44,High,Confirmed high priority,2,2,3,...,84.24,44,119,495,3,3,3,3,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 3,22.82,Low,Low priority,4,0,0,...,100.00,29,62,283,2,3,2,0,0,0
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,22.72,Low,Low priority,2,0,0,...,100.00,21,54,192,1,1,2,0,0,0
117,VND,Vendas e Negociação,Comercial,Centro 3,21.93,Low,Low priority,4,0,0,...,100.00,37,87,371,2,3,3,0,1,0
118,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,21.62,Low,Low priority,4,0,0,...,100.00,16,54,307,1,3,2,0,0,0


## Save results

In [118]:
trainer_requirements_df.to_csv(
    OUTPUTS_DATA_DIR / "trainer_capacity_requirements.csv",
    index=False,
    encoding="utf-8"
)